In [44]:
import json
import pandas as pd
import re
import html

# import raw data

In [45]:
file_paths = [
    '../data/raw/gcp.json',
    '../data/raw/gct.json',
    '../data/raw/gcm.json'
]

df_dict: dict[str, pd.DataFrame] = {}

for path in file_paths:
    with open(path) as f:
        data = json.load(f)

    df = pd.json_normalize(data, max_level=2)
    key = path.split('/')[-1].split('.')[0]
    df_dict[key] = df

for k, v in df_dict.items():
    print(f"{k}: {type(v)} with shape {v.shape}")

gcp: <class 'pandas.core.frame.DataFrame'> with shape (910, 15)
gct: <class 'pandas.core.frame.DataFrame'> with shape (347, 15)
gcm: <class 'pandas.core.frame.DataFrame'> with shape (332, 15)


In [46]:
df_dict['gcp'].columns

Index(['review_id', 'place_id', 'author', 'rating', 'likes', 'user_images',
       'author_profile_url', 'profile_picture', 'created_date', 'review_date',
       'last_modified_date', 'company', 'source', 'description.en',
       'owner_responses.en.text'],
      dtype='object')

In [47]:
# keep essential columns of data + remove personal identifier columns + rename final columns

keep_columns = ['review_id', 'rating', 'likes', 'created_date', 'review_date', 'company', 'source', 'description.en', 'owner_responses.en.text']

for key, frame in df_dict.items():
    df_dict[key] = frame[keep_columns]
    df_dict[key] = df_dict[key].rename(columns={
        'description.en': 'review_text',
        'owner_responses.en.text': 'owner_response_text',
        'created_date': 'date_review_scraped'
    })

In [48]:
df_dict['gcp'].head()

,review_id,rating,likes,date_review_scraped,review_date,company,source,review_text,owner_response_text
0,ChdDSUhNMG9nS0VJQ0FnSURRaXMzbHF3RRAB,4.0,0,2026-07-26T17:13:09.090849+00:00,2013-07-29T17:13:09+00:00,Parmer,Google Maps,As a guy with a non complicated hair cut....th...,NaN
1,ChdDSUhNMG9nS0VJQ0FnSURBNWJXTjdBRRAB,5.0,0,2026-07-26T17:13:08.985569+00:00,2014-07-29T17:13:08+00:00,Parmer,Google Maps,NaN,NaN
2,ChZDSUhNMG9nS0VJQ0FnSUNRb2NhQVRREAE,1.0,2,2026-07-26T17:13:08.891962+00:00,2014-07-29T17:13:08+00:00,Parmer,Google Maps,We have been going to Great Clips for my sons ...,NaN
3,ChdDSUhNMG9nS0VJQ0FnSURBcXZpVnFRRRAB,5.0,0,2026-07-26T17:13:08.662196+00:00,2015-07-29T17:13:08+00:00,Parmer,Google Maps,Our family goes to a sweet girl named Crystal....,NaN
4,ChdDSUhNMG9nS0VJQ0FnSUNnODhYbHFnRRAB,1.0,2,2026-07-26T17:13:08.559022+00:00,2016-07-28T17:13:08+00:00,Parmer,Google Maps,I show up first. Then some idiot signs in onli...,NaN


In [49]:
df_dict['gct'].head()

,review_id,rating,likes,date_review_scraped,review_date,company,source,review_text,owner_response_text
0,ChZDSUhNMG9nS0VJQ0FnSURBcVlXaWJ3EAE,1.0,1,2026-07-26T17:17:51.481416+00:00,2014-07-29T17:17:51+00:00,Techridge,Google Maps,The worst haircut of my life! It was my first...,NaN
1,ChZDSUhNMG9nS0VJQ0FnSURBM0xiMUJ3EAE,5.0,0,2026-07-26T17:17:51.253068+00:00,2015-07-29T17:17:51+00:00,Techridge,Google Maps,Diane just did a cut for me and she was very k...,NaN
2,ChdDSUhNMG9nS0VJQ0FnSUNnMHVxYmt3RRAB,1.0,0,2026-07-26T17:17:51.148891+00:00,2015-07-29T17:17:51+00:00,Techridge,Google Maps,Not good . My sideburns were cut uneven. I wil...,NaN
3,ChdDSUhNMG9nS0VJQ0FnSURBbUtPci13RRAB,5.0,0,2026-07-26T17:17:51.042801+00:00,2016-07-28T17:17:51+00:00,Techridge,Google Maps,NaN,NaN
4,ChdDSUhNMG9nS0VJQ0FnSURBeWVfd3RnRRAB,1.0,0,2026-07-26T17:17:50.954384+00:00,2016-07-28T17:17:50+00:00,Techridge,Google Maps,Worst hair cut I seen before. Waste of money.,NaN


In [50]:
df_dict['gcm'].head()

,review_id,rating,likes,date_review_scraped,review_date,company,source,review_text,owner_response_text
0,ChdDSUhNMG9nS0VJQ0FnSURBaFlUUW5BRRAB,5.0,0,2026-07-26T17:21:40.167897+00:00,2016-07-28T17:21:40+00:00,Manor,Google Maps,NaN,NaN
1,ChZDSUhNMG9nS0VJQ0FnSUN3bVBUX0NnEAE,1.0,0,2026-07-26T17:21:40.077765+00:00,2016-07-28T17:21:40+00:00,Manor,Google Maps,NaN,NaN
2,ChZDSUhNMG9nS0VJQ0FnSURRb2FYZkF3EAE,5.0,0,2026-07-26T17:21:37.530980+00:00,2016-07-28T17:21:37+00:00,Manor,Google Maps,New stylist Brittany is wonderful! She is very...,NaN
3,ChZDSUhNMG9nS0VJQ0FnSURBMnVqVVhBEAE,4.0,0,2026-07-26T17:21:37.431275+00:00,2016-07-28T17:21:37+00:00,Manor,Google Maps,NaN,NaN
4,ChdDSUhNMG9nS0VJQ0FnSUNBZy1qd2t3RRAB,5.0,0,2026-07-26T17:21:37.340656+00:00,2016-07-28T17:21:37+00:00,Manor,Google Maps,NaN,NaN


In [51]:
reviews_df = pd.concat(df_dict.values(), ignore_index=True)
print(reviews_df.shape)
reviews_df.head(10)

(1589, 9)


,review_id,rating,likes,date_review_scraped,review_date,company,source,review_text,owner_response_text
0,ChdDSUhNMG9nS0VJQ0FnSURRaXMzbHF3RRAB,4.0,0,2026-07-26T17:13:09.090849+00:00,2013-07-29T17:13:09+00:00,Parmer,Google Maps,As a guy with a non complicated hair cut....th...,NaN
1,ChdDSUhNMG9nS0VJQ0FnSURBNWJXTjdBRRAB,5.0,0,2026-07-26T17:13:08.985569+00:00,2014-07-29T17:13:08+00:00,Parmer,Google Maps,NaN,NaN
2,ChZDSUhNMG9nS0VJQ0FnSUNRb2NhQVRREAE,1.0,2,2026-07-26T17:13:08.891962+00:00,2014-07-29T17:13:08+00:00,Parmer,Google Maps,We have been going to Great Clips for my sons ...,NaN
3,ChdDSUhNMG9nS0VJQ0FnSURBcXZpVnFRRRAB,5.0,0,2026-07-26T17:13:08.662196+00:00,2015-07-29T17:13:08+00:00,Parmer,Google Maps,Our family goes to a sweet girl named Crystal....,NaN
4,ChdDSUhNMG9nS0VJQ0FnSUNnODhYbHFnRRAB,1.0,2,2026-07-26T17:13:08.559022+00:00,2016-07-28T17:13:08+00:00,Parmer,Google Maps,I show up first. Then some idiot signs in onli...,NaN
5,ChdDSUhNMG9nS0VJQ0FnSURneTQ3SGxBRRAB,1.0,2,2026-07-26T17:13:08.456377+00:00,2016-07-28T17:13:08+00:00,Parmer,Google Maps,Horrible!,NaN
6,ChdDSUhNMG9nS0VJQ0FnSUNBbnFpNnFnRRAB,5.0,0,2026-07-26T17:13:08.349796+00:00,2016-07-28T17:13:08+00:00,Parmer,Google Maps,I have used this location for almost 4 years. ...,NaN
7,ChdDSUhNMG9nS0VJQ0FnSURnNmE3Rl93RRAB,3.0,0,2026-07-26T17:13:08.122797+00:00,2016-07-28T17:13:08+00:00,Parmer,Google Maps,NaN,NaN
8,ChdDSUhNMG9nS0VJQ0FnSUNBOW9XYnV3RRAB,1.0,2,2026-07-26T17:13:08.027830+00:00,2016-07-28T17:13:07+00:00,Parmer,Google Maps,One of worst place they don't know how to cut ...,NaN
9,ChZDSUhNMG9nS0VJQ0FnSURBaXZhbmFREAE,1.0,2,2026-07-26T17:13:07.922222+00:00,2016-07-28T17:13:07+00:00,Parmer,Google Maps,I had my hair cut here yesterday. The woman wh...,NaN


In [52]:
reviews_df.to_csv('../data/raw/combined_review_data.csv', index=False)

# clean data

In [53]:
# Data cleaning steps for Google Maps Reviews dataset

# 1) clean date columns for readability
# 2) encode categorical variables as needed (e.g. location)
# 3) rough clean up the text data but not excessively altering text structure
# 4) handle missing values appropriately
# 5) save cleaned dataframe to new CSV file for analysis

In [54]:
reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1589 entries, 0 to 1588
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   review_id            1589 non-null   object 
 1   rating               1589 non-null   float64
 2   likes                1589 non-null   int64  
 3   date_review_scraped  1589 non-null   object 
 4   review_date          1589 non-null   object 
 5   company              1589 non-null   object 
 6   source               1589 non-null   object 
 7   review_text          893 non-null    object 
 8   owner_response_text  1525 non-null   object 
dtypes: float64(1), int64(1), object(7)
memory usage: 111.9+ KB


In [55]:
# 1) clean date columns for readability
reviews_df['review_date'] = pd.to_datetime(reviews_df['review_date']).dt.strftime('%Y/%m/%d')
reviews_df['date_review_scraped'] = pd.to_datetime(reviews_df['date_review_scraped']).dt.strftime('%Y/%m/%d')

In [56]:
# 2) encode categorial variables as needed (`company` in this case)
unique_companies = reviews_df['company'].unique()
company_mapping = {company: idx+1 for idx, company in enumerate(unique_companies)}

reviews_df['company'] = reviews_df['company'].map(company_mapping)

In [57]:
# 3) rough clean up of text data (e.g. remove special characters, web elements, etc.)
text_cols = ['review_text', 'owner_response_text']

_url_re = re.compile(r'https?://\S+|www\.\S+')
_email_re = re.compile(r'\b[\w\.-]+@[\w\.-]+\.\w+\b')
_ctrl_re = re.compile(r'[\x00-\x1f\x7f-\x9f]')
_multi_space_re = re.compile(r'\s+')

def standardize_text(x, lowercase=False, remove_urls=True, remove_emails=True):
    """
    Remove content from the text such as URLs, emails, and HTML elements
    that don't contain useful information about the text content
    """
    if pd.isna(x):
        return x
    
    x = str(x)
    x = html.unescape(x)
    x = x.replace("\u00a0", " ")
    x = _ctrl_re.sub(" ", x)
    if remove_urls:
        x = _url_re.sub(" ", x)
    if remove_emails:
        x = _email_re.sub(" ", x)
    x = _multi_space_re.sub(" ", x).strip()
    if lowercase:
        x = x.lower()
    return x


for col in text_cols:
    if col in reviews_df.columns:
        reviews_df[col] = reviews_df[col].apply(standardize_text)

In [58]:
# 4) handle missing values appropriately 

'''
2 cases:
    - a customer leaves a review with only a rating and no review description
    - a customer leaves a review with a rating AND a review description

keep all for now
'''
reviews_df[~reviews_df['review_text'].isna()]

,review_id,rating,likes,date_review_scraped,review_date,company,source,review_text,owner_response_text
0,ChdDSUhNMG9nS0VJQ0FnSURRaXMzbHF3RRAB,4.0,0,2026/07/26,2013/07/29,1,Google Maps,As a guy with a non complicated hair cut....th...,NaN
2,ChZDSUhNMG9nS0VJQ0FnSUNRb2NhQVRREAE,1.0,2,2026/07/26,2014/07/29,1,Google Maps,We have been going to Great Clips for my sons ...,NaN
3,ChdDSUhNMG9nS0VJQ0FnSURBcXZpVnFRRRAB,5.0,0,2026/07/26,2015/07/29,1,Google Maps,Our family goes to a sweet girl named Crystal....,NaN
4,ChdDSUhNMG9nS0VJQ0FnSUNnODhYbHFnRRAB,1.0,2,2026/07/26,2016/07/28,1,Google Maps,I show up first. Then some idiot signs in onli...,NaN
5,ChdDSUhNMG9nS0VJQ0FnSURneTQ3SGxBRRAB,1.0,2,2026/07/26,2016/07/28,1,Google Maps,Horrible!,NaN
...,...,...,...,...,...,...,...,...,...
1582,Ci9DQUlRQUNvZENodHljRjlvT2pocFlYRTJiRWxEYTBGcl...,5.0,0,2026/07/26,2026/04/27,3,Google Maps,Rudy was fantastic! He was so happy and energe...,"Hi Melissa, thank you for your wonderful revie..."
1584,Ci9DQUlRQUNvZENodHljRjlvT2tkeVdHWnNUVTFZUmpRM2...,4.0,0,2026/07/26,2026/05/27,3,Google Maps,"Today, I came in because I had limited time fo...","Hi Donna, thank you for sharing your experienc..."
1585,Ci9DQUlRQUNvZENodHljRjlvT25rdFQycHNjVWgyUVRrel...,5.0,0,2026/07/26,2026/05/27,3,Google Maps,Fast friendly service,"Hi Thomas, thank you for your review! It's gre..."
1587,ChdDSUhNMG9nS0VJQ0FnSUMtcTltWHZnRRAB,1.0,0,2026/07/26,2026/06/26,3,Google Maps,I would give zero stars if I could. I used onl...,Thank you for being our customer! We’re always...


In [59]:
reviews_df.to_csv('../data/cleaned/combined_review_data_cleaned.csv', index=False)